# 09 — Retriever
### LangChain Retrieval · Part 4 of 4

`08_Vector_Store/` ended with `.as_retriever()` — the bridge from "a store I can search" to "a
`Runnable` that fits in a chain." Plain similarity search, the default behind that bridge, is the
simplest way to query a vector store. It is not the only way, and for several real failure modes,
not the best one. This notebook covers the retrieval strategies that actually fix those failures.

> 📌 **Worth knowing (LangChain 1.0+, 2026)**
> - `MultiQueryRetriever`, `ContextualCompressionRetriever`, `EnsembleRetriever`, and
>   `ParentDocumentRetriever` all moved out of core LangChain in the 1.0 restructure. They now live in
>   a separate `langchain-classic` package (`pip install langchain-classic`), imported from
>   `langchain_classic.retrievers`. They are not deprecated or broken — "classic" here means
>   "pre-1.0 API, still supported," not "old and going away." If you see `ModuleNotFoundError` for
>   `langchain.retrievers` in any tutorial from before 2026, this is why, and this is the fix.
> - `BM25Retriever` is the odd one out: it stayed in `langchain_community` (the sunset package from
>   `06_Document_Loaders`) rather than moving to `langchain-classic`.

## Setup

In [ ]:
# %pip install -q langchain langchain-core langchain-text-splitters python-dotenv
# %pip install -q langchain-huggingface sentence-transformers langchain-chroma langchain-groq



Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Environment ready.")


## Sample knowledge base

Same support-policy world as `06`–`08`, deliberately including a few chunks that are semantically
*close but wrong* for some queries — plain similarity search handles the easy cases fine; the
interesting failures only show up once there's something for the model to get confused by.

In [ ]:
docs = [
    Document(page_content="Refunds are issued to the original payment method within 5-7 business days after we receive the returned item.", metadata={"department": "billing"}),
    Document(page_content="Shipping fees are non-refundable unless the item arrived damaged or defective.", metadata={"department": "billing"}),
    Document(page_content="Gift cards and digital products cannot be returned once delivered.", metadata={"department": "billing"}),
    Document(page_content="Failed payments give the customer 3 days to update their payment method before automatic cancellation.", metadata={"department": "billing"}),
    Document(page_content="Standard shipping takes 3-5 business days. Orders over $50 qualify for free standard shipping.", metadata={"department": "shipping"}),
    Document(page_content="International shipping takes 7-14 business days and is available to select countries only.", metadata={"department": "shipping"}),
    Document(page_content="Express shipping takes 1-2 business days for orders placed before 2 PM.", metadata={"department": "shipping"}),
    Document(page_content="To reset your password, choose 'Forgot password' on the sign-in page and follow the emailed link.", metadata={"department": "account"}),
    Document(page_content="Two-factor authentication codes are sent by SMS and may take up to 2 minutes to arrive.", metadata={"department": "account"}),
    Document(page_content="You can change your account email from Settings > Profile > Email Address.", metadata={"department": "account"}),
]

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings, collection_name="support_retriever_kb")
print(f"{vectorstore._collection.count()} documents indexed")


---
## 1. Plain similarity search — the baseline, and where it struggles

`.as_retriever()` with no `search_type` set defaults to plain similarity search — exactly what
`08_Vector_Store` already covered. Its weak spot: **returning only the single closest match can miss
genuinely useful results that happen to be phrased slightly differently**, and it has no mechanism to
avoid returning several near-duplicate chunks instead of a spread of distinct, useful ones.

In [ ]:
similarity_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = similarity_retriever.invoke("what if a payment fails")
for d in results:
    print(d.metadata["department"], "->", d.page_content)


---
## 2. MMR — Maximal Marginal Relevance: relevant, but not all the same

Ask for `k=3` billing-related results with plain similarity search, and you can easily get three
chunks that are all slightly different phrasings of the *same* point — because they're all close to
the query, which is exactly what similarity search optimizes for. **MMR** optimizes for something
different: relevant **and** different from each other, so your `k` results actually cover more
ground instead of repeating the same answer three times.

`lambda_mult` controls the trade-off: `1.0` = pure relevance (behaves like plain similarity search),
`0.0` = pure diversity (relevance barely matters). `0.5` is a reasonable starting point.

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 8, "lambda_mult": 0.5},   # fetch_k: how many candidates MMR picks FROM
)

print("--- similarity (k=3) ---")
for d in similarity_retriever.invoke("billing question"):
    print(" -", d.page_content[:70])

print("\n--- MMR (k=3, lambda_mult=0.5) ---")
for d in mmr_retriever.invoke("billing question"):
    print(" -", d.page_content[:70])


With a generic query like "billing question," MMR is more likely to surface one refund chunk, one
failed-payment chunk, and one gift-card chunk — a spread across the topic — rather than three
near-duplicate "billing-adjacent" matches.

---
## 3. `MultiQueryRetriever` — fixing queries that are too narrow

A single embedded query only searches from *one angle*. A customer who writes "money back" won't
necessarily retrieve a chunk about "refunds" as strongly as a customer who happened to use that exact
word. `MultiQueryRetriever` asks an LLM to generate **several reworded versions** of the question,
retrieves for each one, and merges the unique results — covering more angles than any single phrasing.

In [ ]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm=llm,
)

results = multi_query_retriever.invoke("how do I get my money back")
print(f"{len(results)} unique documents retrieved across all reworded queries:")
for d in results:
    print(" -", d.page_content[:70])


`MultiQueryRetriever` makes several LLM calls just to generate rewordings before it even searches
— a real cost and latency trade-off. Reach for it when queries are often short, vague, or phrased
very differently from your documents' wording — a support chatbot fielding real customer phrasing is
exactly that case.

---
## 4. `EnsembleRetriever` — hybrid search: meaning AND exact words

Vector search finds *meaning* and can genuinely miss **exact terms** — a product SKU, an error code,
an acronym, a person's name — because embeddings aren't optimized for exact-string matching.
Classic keyword search (**BM25**) is the opposite: great at exact terms, blind to synonyms and
rephrasing. `EnsembleRetriever` runs both and combines their rankings — the actual production answer
to "vector search missed the exact phrase the customer typed." 

In [ ]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 3

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vectorstore.as_retriever(search_kwargs={"k": 3})],
    weights=[0.5, 0.5],   # equal trust in exact-keyword and semantic signals
)

query = "2FA SMS code"   # an exact, jargon-heavy phrase — the case BM25 is built for

print("--- vector-only ---")
for d in vectorstore.as_retriever(search_kwargs={"k": 2}).invoke(query):
    print(" -", d.page_content[:70])

print("\n--- BM25-only ---")
for d in bm25_retriever.invoke(query):
    print(" -", d.page_content[:70])

print("\n--- Ensemble (hybrid) ---")
for d in ensemble_retriever.invoke(query):
    print(" -", d.page_content[:70])


---
## 5. `ContextualCompressionRetriever` — trimming what you retrieve down to what's relevant

A retrieved chunk is often *mostly* relevant, with a sentence or two of noise riding along. Sending
the whole chunk to the final answering prompt wastes tokens and can dilute the model's focus.
`ContextualCompressionRetriever` wraps a base retriever with an LLM pass that **extracts only the
relevant sentences** from each retrieved chunk before handing it onward.

> ⚠️ Reference-leaning section: this makes one extra LLM call *per retrieved document*, on every
> single query. For a 3-document retrieval, that's 3 extra calls before you've even generated an
> answer. Worth knowing it exists; worth being deliberate about whether the latency/cost trade-off is
> worth it for a given use case.

In [ ]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
)

results = compression_retriever.invoke("what happens if my payment fails")
for d in results:
    print(repr(d.page_content))


---
## 6. `ParentDocumentRetriever` — small chunks for searching, big chunks for answering

A real tension from `07_Text_Splitters`: **small chunks embed and match better** (a focused chunk's
vector isn't diluted by unrelated neighboring sentences), but **small chunks often lack the context
a model needs to answer well**. `ParentDocumentRetriever` resolves this by splitting twice: small
"child" chunks get embedded and searched, but when a child matches, its **larger parent chunk** —
with more surrounding context — is what actually gets returned.

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)  # bigger than our one sample doc, on purpose
child_splitter = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=0)

parent_store = Chroma(collection_name="parent_demo", embedding_function=embeddings)
docstore = InMemoryStore()

parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=parent_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

long_policy = Document(
    page_content=(
        "Refund Policy. Customers may return most items within 30 days of delivery, provided "
        "the item is unused and in its original packaging. Refunds are issued to the original "
        "payment method within 5-7 business days after the returned item is received. Shipping "
        "fees are non-refundable unless the item arrived damaged or defective. Digital products "
        "and gift cards cannot be returned once delivered, with no exceptions."
    ),
    metadata={"source": "returns_policy.txt"},
)
parent_doc_retriever.add_documents([long_policy])   # internally: splits into small children for search,
                                                       # splits into one or more LARGER parents for answering

results = parent_doc_retriever.invoke("digital products refund")
print(f"Original document: {len(long_policy.page_content)} characters")
print(f"{len(results)} parent document(s) returned (parent_splitter chunk_size=600, "
      f"bigger than our one sample doc, so it stays whole):\n")
for d in results:
    print(d.page_content)   # the FULL parent — not the small 60-character child chunk that actually matched


---
## Comparison: which retriever, when

| Retriever | Fixes | Cost |
|---|---|---|
| Plain similarity | Nothing extra — the default | Baseline |
| MMR | Near-duplicate results for broad/generic queries | Free — same embedding search, different selection logic |
| `MultiQueryRetriever` | Queries phrased very differently from your documents | +1 LLM call to generate rewordings |
| `EnsembleRetriever` (BM25 + vector) | Exact terms (codes, names, SKUs) that embeddings miss | Small — BM25 is cheap, no extra LLM call |
| `ContextualCompressionRetriever` | Retrieved chunks are mostly-relevant with noise riding along | +1 LLM call **per retrieved document, per query** |
| `ParentDocumentRetriever` | Small chunks match well but lack context to answer from | Slightly more storage (child index + parent docstore) |

**In practice:** start with MMR — it's free. Add `EnsembleRetriever` the moment your domain has exact
terms worth matching on (product codes, error messages, names). Reach for `MultiQueryRetriever` and
`ContextualCompressionRetriever` deliberately, once you've confirmed the cost is worth it for your
specific failure mode — they're the two options here that multiply your LLM calls.

---
## Real-world capstone: a tuned support retriever, end to end

Combining what actually matters for a support knowledge base: **ensemble** (catch exact terms like
error codes) wrapped as the base for **MMR-style diversity** isn't directly composable in one line,
so the realistic production pattern is: pick the ONE or TWO techniques your failure analysis actually
justifies, not all five at once. Here: hybrid search for a knowledge base where customers often type
exact product/feature names, feeding straight into an answer chain.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(documents):
    return "\n\n".join(d.page_content for d in documents)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. If it's not there, say you don't have that information.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

support_rag_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(ensemble_retriever.invoke(x["question"])))
    | answer_prompt
    | llm
    | StrOutputParser()
)

print(support_rag_chain.invoke({"question": "What's the deal with 2FA SMS codes?"}))


### Try it yourself
1. Run the same query through `similarity_retriever`, `mmr_retriever`, and `ensemble_retriever`
   individually and compare which documents each one returns.
2. Lower `lambda_mult` to `0.1` on the MMR retriever and see how aggressively it favors diversity over
   raw relevance.
3. Change the `weights` on `EnsembleRetriever` to `[0.8, 0.2]` (favor BM25 heavily) and see how the
   results shift for a very conceptual, non-exact query like "what if I'm unhappy with an item."
4. Time `ContextualCompressionRetriever.invoke()` against plain `similarity_retriever.invoke()` on the
   same query — put a number on the latency cost from Section 5's warning.

---
**Next:** `10_RAG_Project/` — every piece from `06` through this notebook, wired into one real,
working "chat with your documents" application with conversational memory.
